# YOLOv11 Training for Axial T2 MRI Images - RSNA 2024 Lumbar Spine Dataset

## Complete End-to-End Training Pipeline

**Based on:** "YOLOv11 Based Classification of Lumbar Spine Degenerative Changes Across Multi-Modal Imaging" (Patel et al., 2025)

**Dataset:** RSNA 2024 Lumbar Spine Degenerative Classification

**Model:** YOLOv11x (Extra Large)

**Focus:** Axial T2 MRI Images Only

**Classes:**
- Class 0: Spinal Canal Stenosis (Center)
- Class 1: Left Neural Foraminal Narrowing & Left Subarticular Stenosis (Left)
- Class 2: Right Neural Foraminal Narrowing & Right Subarticular Stenosis (Right)

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install ultralytics pydicom albumentations -q

# Verify installations
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import cv2
import pydicom
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import yaml
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## 2. Data Filtering - Extract Axial T2 Series Only

In [ ]:
# Define paths
INPUT_PATH = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification'
WORKING_PATH = '/kaggle/working'
DATASET_PATH = os.path.join(WORKING_PATH, 'datasets', 'axial_t2')

# Create directory structure
for split in ['train', 'val']:
    for folder in ['images', 'labels']:
        os.makedirs(os.path.join(DATASET_PATH, split, folder), exist_ok=True)

print(f"Dataset directory structure created at: {DATASET_PATH}")

In [ ]:
# Load series descriptions and filter for Axial T2
series_desc_path = os.path.join(INPUT_PATH, 'train_series_descriptions.csv')
series_df = pd.read_csv(series_desc_path)

# Filter for Axial T2 only
axial_t2_series = series_df[series_df['series_description'] == 'Axial T2'].copy()

print(f"Total series in dataset: {len(series_df)}")
print(f"Axial T2 series found: {len(axial_t2_series)}")
print(f"\nFirst few Axial T2 series:")
print(axial_t2_series.head())

In [ ]:
# Load coordinate labels
coords_path = os.path.join(INPUT_PATH, 'train_label_coordinates.csv')
coords_df = pd.read_csv(coords_path)

print(f"Total coordinate labels: {len(coords_df)}")
print(f"\nCoordinates DataFrame columns:")
print(coords_df.columns.tolist())
print(f"\nSample coordinate data:")
print(coords_df.head())

## 3. Class Mapping Configuration

In [ ]:
# Define class mapping based on paper methodology
# Mapping RSNA conditions to 3 anatomical classes for Axial view

CLASS_MAPPING = {
    # Class 0: Center - Spinal Canal
    'spinal_canal_stenosis': 0,
    
    # Class 1: Left side - Neural Foraminal and Subarticular
    'left_neural_foraminal_narrowing': 1,
    'left_subarticular_stenosis': 1,
    
    # Class 2: Right side - Neural Foraminal and Subarticular
    'right_neural_foraminal_narrowing': 2,
    'right_subarticular_stenosis': 2
}

CLASS_NAMES = ['Center', 'Left', 'Right']

# Bounding box parameters (relative to original image dimensions)
BOX_SIZE = 32  # Fixed box size in pixels
TARGET_SIZE = 384  # Target image size as per paper

print("Class Mapping Configuration:")
for condition, class_id in CLASS_MAPPING.items():
    print(f"  {condition}: Class {class_id} ({CLASS_NAMES[class_id]})")

## 4. DICOM Processing and Label Creation Functions

In [ ]:
def load_dicom_image(dcm_path):
    """
    Load DICOM file and convert to normalized 8-bit image.
    
    Args:
        dcm_path: Path to DICOM file
    
    Returns:
        Normalized 8-bit numpy array
    """
    try:
        # Read DICOM file
        dcm = pydicom.dcmread(dcm_path)
        img = dcm.pixel_array
        
        # Normalize to 0-255 range
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255).astype(np.uint8)
        
        # Convert to 3-channel if grayscale
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        return img, dcm.Rows, dcm.Columns
    except Exception as e:
        print(f"Error loading {dcm_path}: {e}")
        return None, None, None


def create_yolo_bbox(x, y, orig_height, orig_width, box_size=32):
    """
    Convert point coordinates to YOLO format bounding box.
    
    Args:
        x, y: Point coordinates
        orig_height, orig_width: Original image dimensions
        box_size: Fixed box size in pixels
    
    Returns:
        YOLO format bbox: (x_center, y_center, width, height) normalized to 0-1
    """
    # Create fixed-size box centered on point
    half_box = box_size / 2
    
    x_min = max(0, x - half_box)
    y_min = max(0, y - half_box)
    x_max = min(orig_width, x + half_box)
    y_max = min(orig_height, y + half_box)
    
    # Calculate center and dimensions
    bbox_width = x_max - x_min
    bbox_height = y_max - y_min
    x_center = x_min + bbox_width / 2
    y_center = y_min + bbox_height / 2
    
    # Normalize to 0-1
    x_center_norm = x_center / orig_width
    y_center_norm = y_center / orig_height
    width_norm = bbox_width / orig_width
    height_norm = bbox_height / orig_height
    
    return x_center_norm, y_center_norm, width_norm, height_norm


print("DICOM processing functions defined successfully!")

## 5. Dataset Preparation - Process Images and Create Labels

In [ ]:
def process_dataset(axial_t2_series, coords_df, output_base_path, target_size=384):
    """
    Process all Axial T2 images and create YOLO format dataset.
    Implements label propagation to adjacent slices as per paper.
    
    Args:
        axial_t2_series: DataFrame of Axial T2 series
        coords_df: DataFrame with coordinate labels
        output_base_path: Base path for output dataset
        target_size: Target image size (384x384)
    
    Returns:
        List of processed image paths
    """
    processed_images = []
    
    # Group by study_id and series_id
    grouped = axial_t2_series.groupby(['study_id', 'series_id'])
    
    print(f"Processing {len(grouped)} Axial T2 series...")
    
    for (study_id, series_id), group in tqdm(grouped, desc="Processing series"):
        # Find corresponding images
        series_path = os.path.join(INPUT_PATH, 'train_images', str(study_id), str(series_id))
        
        if not os.path.exists(series_path):
            continue
        
        # Get all DICOM files in series
        dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith('.dcm')])
        
        if len(dcm_files) == 0:
            continue
        
        # Get labels for this series
        series_labels = coords_df[
            (coords_df['study_id'] == study_id) & 
            (coords_df['series_id'] == series_id)
        ]
        
        # Create slice-to-labels mapping
        slice_labels = {}
        
        for _, label_row in series_labels.iterrows():
            instance_number = label_row.get('instance_number', None)
            if pd.isna(instance_number):
                continue
            
            instance_number = int(instance_number)
            
            if instance_number not in slice_labels:
                slice_labels[instance_number] = []
            
            # Get condition and coordinates
            condition = label_row.get('condition', '')
            x = label_row.get('x', None)
            y = label_row.get('y', None)
            
            if pd.notna(x) and pd.notna(y) and condition in CLASS_MAPPING:
                class_id = CLASS_MAPPING[condition]
                slice_labels[instance_number].append({
                    'class_id': class_id,
                    'x': float(x),
                    'y': float(y)
                })
        
        # Process each DICOM file
        for dcm_file in dcm_files:
            dcm_path = os.path.join(series_path, dcm_file)
            instance_number = int(dcm_file.replace('.dcm', ''))
            
            # Load and process image
            img, orig_h, orig_w = load_dicom_image(dcm_path)
            
            if img is None:
                continue
            
            # Resize image to target size
            img_resized = cv2.resize(img, (target_size, target_size))
            
            # Collect labels for this slice (including propagated from adjacent slices)
            all_labels = []
            
            # Label propagation strategy: use labels from n-1, n, n+1
            for offset in [-1, 0, 1]:
                check_instance = instance_number + offset
                if check_instance in slice_labels:
                    all_labels.extend(slice_labels[check_instance])
            
            # Create unique filename
            img_filename = f"{study_id}_{series_id}_{instance_number}.jpg"
            label_filename = f"{study_id}_{series_id}_{instance_number}.txt"
            
            # Save image
            img_save_path = os.path.join(output_base_path, 'images', img_filename)
            cv2.imwrite(img_save_path, img_resized)
            
            # Save labels in YOLO format
            label_save_path = os.path.join(output_base_path, 'labels', label_filename)
            
            with open(label_save_path, 'w') as f:
                for label in all_labels:
                    bbox = create_yolo_bbox(
                        label['x'], label['y'], 
                        orig_h, orig_w, 
                        box_size=BOX_SIZE
                    )
                    f.write(f"{label['class_id']} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}\n")
            
            processed_images.append(img_filename)
    
    return processed_images

print("Dataset processing function defined!")

In [ ]:
# Process all Axial T2 images to a temporary location
temp_output = os.path.join(WORKING_PATH, 'temp_processed')
os.makedirs(os.path.join(temp_output, 'images'), exist_ok=True)
os.makedirs(os.path.join(temp_output, 'labels'), exist_ok=True)

processed_images = process_dataset(
    axial_t2_series, 
    coords_df, 
    temp_output, 
    target_size=TARGET_SIZE
)

print(f"\nTotal processed images: {len(processed_images)}")

## 6. Train/Validation Split (80/20)

In [ ]:
# Split dataset into train and validation (80/20)
train_images, val_images = train_test_split(
    processed_images, 
    test_size=0.2, 
    random_state=42
)

print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")

# Move files to train/val directories
for img_name in tqdm(train_images, desc="Moving train files"):
    label_name = img_name.replace('.jpg', '.txt')
    
    # Copy image
    shutil.copy(
        os.path.join(temp_output, 'images', img_name),
        os.path.join(DATASET_PATH, 'train', 'images', img_name)
    )
    
    # Copy label
    shutil.copy(
        os.path.join(temp_output, 'labels', label_name),
        os.path.join(DATASET_PATH, 'train', 'labels', label_name)
    )

for img_name in tqdm(val_images, desc="Moving validation files"):
    label_name = img_name.replace('.jpg', '.txt')
    
    # Copy image
    shutil.copy(
        os.path.join(temp_output, 'images', img_name),
        os.path.join(DATASET_PATH, 'val', 'images', img_name)
    )
    
    # Copy label
    shutil.copy(
        os.path.join(temp_output, 'labels', label_name),
        os.path.join(DATASET_PATH, 'val', 'labels', label_name)
    )

# Clean up temporary directory
shutil.rmtree(temp_output)

print("\nDataset split completed!")

## 7. Data Augmentation Configuration

In [ ]:
# Define augmentation pipeline as per paper
# Rotation, Scaling, and Flipping

augmentation_pipeline = A.Compose([
    A.Rotate(limit=15, p=0.5),  # Random rotation up to 15 degrees
    A.RandomScale(scale_limit=0.2, p=0.5),  # Random scaling
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.3),  # Vertical flip
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print("Augmentation pipeline configured:")
print("  - Rotation (±15 degrees)")
print("  - Random Scaling (±20%)")
print("  - Horizontal Flip")
print("  - Vertical Flip")
print("\nNote: YOLO's built-in augmentation will be used during training.")

## 8. Create data.yaml Configuration File

In [ ]:
# Create data.yaml for YOLO training
data_yaml = {
    'path': DATASET_PATH,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 3,  # Number of classes
    'names': CLASS_NAMES  # ['Center', 'Left', 'Right']
}

yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml created at: {yaml_path}")
print("\nConfiguration:")
print(yaml.dump(data_yaml, default_flow_style=False))

## 9. Visualize Sample Training Data

In [ ]:
def visualize_sample(img_path, label_path, class_names):
    """
    Visualize a training sample with bounding boxes.
    """
    # Load image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Load labels
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img)
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    x_center, y_center, width, height = map(float, parts[1:])
                    
                    # Convert YOLO format to pixel coordinates
                    x_center_px = x_center * w
                    y_center_px = y_center * h
                    width_px = width * w
                    height_px = height * h
                    
                    x_min = x_center_px - width_px / 2
                    y_min = y_center_px - height_px / 2
                    
                    # Draw bounding box
                    rect = patches.Rectangle(
                        (x_min, y_min), width_px, height_px,
                        linewidth=2, edgecolor='red', facecolor='none'
                    )
                    ax.add_patch(rect)
                    
                    # Add label
                    ax.text(
                        x_min, y_min - 5,
                        class_names[class_id],
                        color='red', fontsize=12, weight='bold',
                        bbox=dict(facecolor='white', alpha=0.7)
                    )
    
    ax.axis('off')
    plt.title('Sample Training Image with Annotations', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()

# Visualize a few training samples
train_img_dir = os.path.join(DATASET_PATH, 'train', 'images')
train_label_dir = os.path.join(DATASET_PATH, 'train', 'labels')

sample_images = [f for f in os.listdir(train_img_dir) if f.endswith('.jpg')][:3]

for img_file in sample_images:
    img_path = os.path.join(train_img_dir, img_file)
    label_path = os.path.join(train_label_dir, img_file.replace('.jpg', '.txt'))
    visualize_sample(img_path, label_path, CLASS_NAMES)

## 10. Initialize YOLOv11x Model

In [ ]:
# Initialize YOLOv11x (Extra Large) model
# The model will be automatically downloaded if not present

model = YOLO('yolo11x.pt')  # YOLOv11 Extra Large pretrained model

print("YOLOv11x model initialized successfully!")
print(f"\nModel architecture: YOLOv11x")
print(f"Parameters: Extra Large configuration")

## 11. Model Training with Paper Hyperparameters

### Training Configuration (Table III from Paper):
- **Image Size:** 384x384
- **Batch Size:** 16
- **Epochs:** 50 (reduced from 100-120 for Kaggle time limits)
- **Optimizer:** AdamW
- **Learning Rate:** 0.001
- **Weight Decay:** 0.0005
- **Dropout:** 0.2
- **Patience:** 15
- **LR Scheduler:** Cosine Annealing
- **Warmup Epochs:** 10
- **Momentum:** 0.8

In [ ]:
# Train the model with exact hyperparameters from the paper
results = model.train(
    data=yaml_path,
    imgsz=384,                    # Image size (384x384)
    batch=16,                     # Batch size
    epochs=50,                    # Number of epochs (50 for Kaggle, paper used 100-120)
    optimizer='AdamW',            # AdamW optimizer
    lr0=0.001,                    # Initial learning rate
    weight_decay=0.0005,          # Weight decay
    dropout=0.2,                  # Dropout rate
    patience=15,                  # Early stopping patience
    cos_lr=True,                  # Cosine learning rate scheduler
    warmup_epochs=10,             # Warmup epochs
    momentum=0.8,                 # SGD momentum (also affects AdamW beta1)
    save=True,                    # Save checkpoints
    save_period=10,               # Save every 10 epochs
    project=os.path.join(WORKING_PATH, 'yolo11_axial_t2'),  # Project directory
    name='train',                 # Experiment name
    exist_ok=True,                # Overwrite existing
    pretrained=True,              # Use pretrained weights
    verbose=True,                 # Verbose output
    seed=42,                      # Random seed for reproducibility
    deterministic=True,           # Deterministic mode
    # Augmentation settings (as per paper)
    hsv_h=0.015,                  # HSV-Hue augmentation
    hsv_s=0.7,                    # HSV-Saturation augmentation
    hsv_v=0.4,                    # HSV-Value augmentation
    degrees=15.0,                 # Rotation degrees
    translate=0.1,                # Translation
    scale=0.2,                    # Scale augmentation
    shear=0.0,                    # Shear augmentation
    perspective=0.0,              # Perspective augmentation
    flipud=0.3,                   # Vertical flip probability
    fliplr=0.5,                   # Horizontal flip probability
    mosaic=1.0,                   # Mosaic augmentation
    mixup=0.0,                    # Mixup augmentation
)

print("\nTraining completed!")
print(f"Best model saved at: {os.path.join(WORKING_PATH, 'yolo11_axial_t2', 'train', 'weights', 'best.pt')}")

## 12. Display Training Results

In [ ]:
# Display training metrics
results_dir = os.path.join(WORKING_PATH, 'yolo11_axial_t2', 'train')

# Show training curves
results_img = os.path.join(results_dir, 'results.png')
if os.path.exists(results_img):
    img = plt.imread(results_img)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Results image not found.")

# Show confusion matrix
confusion_matrix_img = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(confusion_matrix_img):
    img = plt.imread(confusion_matrix_img)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix image not found.")

## 13. Model Evaluation on Validation Set

In [ ]:
# Load best model
best_model_path = os.path.join(results_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

print(f"Best model loaded from: {best_model_path}")

# Evaluate on validation set
val_results = best_model.val(
    data=yaml_path,
    imgsz=384,
    batch=16,
    save_json=True,
    save_hybrid=True,
    conf=0.25,
    iou=0.6,
    max_det=100,
    plots=True
)

print("\nValidation Metrics:")
print(f"  mAP50: {val_results.box.map50:.4f}")
print(f"  mAP50-95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall: {val_results.box.mr:.4f}")

## 14. Inference and Visualization (Similar to Figure 9 in Paper)

In [ ]:
def visualize_predictions(model, image_path, class_names, conf_threshold=0.25):
    """
    Run inference and visualize predictions on MRI image.
    Mimics Figure 9 from the paper.
    
    Args:
        model: Trained YOLO model
        image_path: Path to image
        class_names: List of class names
        conf_threshold: Confidence threshold
    """
    # Run inference
    results = model(image_path, conf=conf_threshold, imgsz=384)
    
    # Load original image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    ax.imshow(img)
    
    # Get predictions
    for result in results:
        boxes = result.boxes
        
        for box in boxes:
            # Get box coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls = int(box.cls[0].cpu().numpy())
            
            # Draw bounding box
            width = x2 - x1
            height = y2 - y1
            
            # Color based on class
            colors = ['yellow', 'cyan', 'magenta']
            color = colors[cls] if cls < len(colors) else 'red'
            
            rect = patches.Rectangle(
                (x1, y1), width, height,
                linewidth=3, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            
            # Add label with confidence
            label = f"{class_names[cls]}: {conf:.2f}"
            ax.text(
                x1, y1 - 10,
                label,
                color='white', fontsize=12, weight='bold',
                bbox=dict(facecolor=color, alpha=0.8, edgecolor='white', linewidth=2)
            )
    
    ax.axis('off')
    plt.title('YOLOv11 Predictions on Axial T2 MRI', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()

# Visualize predictions on validation samples
val_img_dir = os.path.join(DATASET_PATH, 'val', 'images')
val_images = [f for f in os.listdir(val_img_dir) if f.endswith('.jpg')][:5]

print("Visualizing predictions on validation samples...\n")

for img_file in val_images:
    img_path = os.path.join(val_img_dir, img_file)
    print(f"Image: {img_file}")
    visualize_predictions(best_model, img_path, CLASS_NAMES, conf_threshold=0.25)
    print("-" * 80)

## 15. Export Model for Inference

In [ ]:
# Export model to different formats for deployment
export_formats = ['onnx', 'torchscript']

for fmt in export_formats:
    try:
        export_path = best_model.export(format=fmt, imgsz=384)
        print(f"Model exported to {fmt.upper()}: {export_path}")
    except Exception as e:
        print(f"Failed to export to {fmt}: {e}")

print("\nModel export completed!")

## 16. Summary and Final Results

In [ ]:
print("="*80)
print("YOLOV11 AXIAL T2 MRI TRAINING - SUMMARY")
print("="*80)
print("\n📊 Dataset Statistics:")
print(f"  Total Axial T2 Series: {len(axial_t2_series)}")
print(f"  Training Images: {len(train_images)}")
print(f"  Validation Images: {len(val_images)}")
print(f"  Image Size: {TARGET_SIZE}x{TARGET_SIZE}")

print("\n🎯 Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  Class {i}: {name}")

print("\n⚙️ Model Configuration:")
print(f"  Architecture: YOLOv11x (Extra Large)")
print(f"  Image Size: 384x384")
print(f"  Batch Size: 16")
print(f"  Epochs: 50")
print(f"  Optimizer: AdamW")
print(f"  Learning Rate: 0.001")
print(f"  Weight Decay: 0.0005")
print(f"  Dropout: 0.2")

print("\n📈 Training Results:")
print(f"  Best Model: {best_model_path}")
print(f"  mAP50: {val_results.box.map50:.4f}")
print(f"  mAP50-95: {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall: {val_results.box.mr:.4f}")

print("\n✅ Training Complete!")
print(f"\n📁 All results saved to: {results_dir}")
print("="*80)

## Conclusion

This notebook implements a complete end-to-end pipeline for training YOLOv11x on Axial T2 MRI images from the RSNA 2024 Lumbar Spine dataset, following the methodology described in "YOLOv11 Based Classification of Lumbar Spine Degenerative Changes Across Multi-Modal Imaging" (Patel et al., 2025).

### Key Features Implemented:
1. ✅ Data filtering for Axial T2 images only
2. ✅ DICOM processing with normalization to 8-bit
3. ✅ Resizing to 384x384 pixels
4. ✅ YOLO bounding box conversion from coordinate points
5. ✅ 3-class mapping (Center, Left, Right)
6. ✅ Label propagation to adjacent slices (n-1, n, n+1)
7. ✅ Data augmentation (Rotation, Scaling, Flipping)
8. ✅ YOLOv11x model training
9. ✅ Exact hyperparameters from Table III (paper)
10. ✅ Evaluation and visualization (Figure 9 style)

### Next Steps:
- Fine-tune hyperparameters for improved performance
- Extend to other MRI modalities (Sagittal T1, Sagittal T2)
- Implement ensemble methods
- Deploy model for clinical inference